# 1 - Aplicação real - Transição de Sistemas e Pipeline de Limpeza

Contexto: A nossa varejista está modernizando a infraestrutura. Metade das nossas vendas ainda vem do sistema antigo (que gera textos bagunçados) e a outra metade já vem da nova API (que gera dicionários estruturados).
Como Analistas de Dados, vocês precisam unificar tudo isso. O script de vocês deve ler cada registro, identificar de qual sistema ele veio, extrair as informações corretamente e, por fim, passar tudo por uma mesma regra de precificação.
As Regras de Negócio:

Identificação da Fonte:
Se o registro for um Dicionário: Verifiquem a chave "status". Se for "erro", ignorem a linha. Se for "ok", extraiam o produto, quantidade e preço de suas respectivas chaves.

Se o registro for uma String: Apliquem as regras antigas (pular linhas que começam com "ERRO" ou "TESTE", trocar ";" por ",", limpar espaços e separar os dados). Se não tiver os 3 dados, descartar.

Regra de Precificação Única (if / elif / else):

Independentemente de onde o dado veio, calcule o Subtotal (Qtd x Preço).
> R$ 1000: Venda VIP (10% de desconto).
Entre R$ 100 e R$ 1000: Venda Padrão (Frete Grátis, valor igual).
< R$ 100: Venda Varejo (+ R$ 15 de frete).

## Dados de entrada - simulação

logs_do_sistema = [
    "ERRO: Banco de dados desconectado as 03:00",
    {"produto": "Placa de Video RTX", "quantidade": 1, "preco": 3500.00, "status": "ok"},
    "   mouse gamer  ,  5 ,  45.90   ",
    "TESTE: admin, 1, 0.00",
    {"status": "erro", "mensagem": "Timeout no gateway de pagamento"},
    "Monitor Ultrawide; 1; 1500.00",
    {"produto": "Pendrive 32GB", "quantidade": 3, "preco": 25.00, "status": "ok"},
    "Cadeira de Escritorio, 2"
]


In [4]:
#Dados
logs_do_sistema = [
    "ERRO: Banco de dados desconectado as 03:00",
    {"produto": "Placa de Video RTX", "quantidade": 1, "preco": 3500.00, "status": "ok"},
    "   mouse gamer  ,  5 ,  45.90   ",
    "TESTE: admin, 1, 0.00",
    {"status": "erro", "mensagem": "Timeout no gateway de pagamento"},
    "Monitor Ultrawide; 1; 1500.00",
    {"produto": "Pendrive 32GB", "quantidade": 3, "preco": 25.00, "status": "ok"},
    "Cadeira de Escritorio, 2"
]

#Percorrer cada registro
for registro in logs_do_sistema:

    if isinstance(registro, dict): #Veio do API

        if registro["status"] == "erro": #Verificar status
            continue

        produto = registro["produto"]
        quantidade = registro["quantidade"]
        preco = registro["preco"]

    elif isinstance(registro, str): #Veio do ambiente antigo

        if registro.startswith("ERRO") or registro.startswith("TESTE"): #verificar erro
            continue

        registro = registro.replace(";", ",") #Trocar ; por ,
        registro = registro.strip() #Limpar espaços

        dados = registro.split(",") #separar dados

        #verificar se existe 3 dados necessários

        if len(dados) != 3:
            continue

        produto = dados[0].strip()
        quantidade = int(dados[1].strip())
        preco = float(dados[2].strip())

    #Igual para dic e string
    subtotal = quantidade * preco

    if subtotal > 1000:
        tipo_venda = "Venda VIP"
        valor_final = subtotal * 0.90

    elif subtotal >= 100:
        tipo_venda = "Venda Padrão"
        valor_final = subtotal

    else:
        tipo_venda = "Venda Varejo"
        valor_final = subtotal + 15

    print(f"Produto: {produto}")
    print(f"Quantidade: {quantidade}")
    print(f"Subtotal: R$ {subtotal:.2f}")
    print(f"Tipo: {tipo_venda}")
    print(f"Valor final: R$ {valor_final:.2f}")
    print("-" * 30)

Produto: Placa de Video RTX
Quantidade: 1
Subtotal: R$ 3500.00
Tipo: Venda VIP
Valor final: R$ 3150.00
------------------------------
Produto: mouse gamer
Quantidade: 5
Subtotal: R$ 229.50
Tipo: Venda Padrão
Valor final: R$ 229.50
------------------------------
Produto: Monitor Ultrawide
Quantidade: 1
Subtotal: R$ 1500.00
Tipo: Venda VIP
Valor final: R$ 1350.00
------------------------------
Produto: Pendrive 32GB
Quantidade: 3
Subtotal: R$ 75.00
Tipo: Venda Varejo
Valor final: R$ 90.00
------------------------------
